In [1]:
# -*- coding: utf-8 -*-
"""RAGTEST.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1zTuU_my84tuxonD304LIWX5Rqf8fchf3
"""

# 1. 安装 LlamaIndex 和相关依赖
!pip install llama-index llama-index-llms-openrouter llama-index-embeddings-huggingface pypdf llama-index-readers-file

import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 2. 配置你的 API Key（把你的 Key 放这里）
# 注意：OpenRouter 要求传入一个特定的 base_url 来重定向请求
from google.colab import userdata
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

# 3. 初始化大模型 (LLM) 和 向量模型 (Embedding)
# 换成 claude-3.5-haiku，理解和总结能力比 8B 模型强很多，价格依然便宜
try:
    llm = OpenRouter(
    api_key=OPENROUTER_API_KEY,
    model="openai/gpt-4o-mini"
    )
    Settings.llm = llm
except Exception as e:
    print(f"LLM 初始化失败，检查 Key 是否有效: {e}")

# 免费的本地向量模型，负责把 PDF 文字变数字，直接在 Colab 内存里跑，不需要消耗 API 额度
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("基础模型配置完成！开始读取数据...")

# 4. 读取刚才上传到 data 文件夹的 PDF
documents = SimpleDirectoryReader("data").load_data()

# === 诊断：检查 PDF 是否被正确读取 ===
print(f"\n[诊断] 共读取到 {len(documents)} 个文档片段")
if len(documents) == 0:
    print("[诊断] 警告：没有读到任何内容，检查 data 文件夹里是否有文件，以及 PDF 是否是扫描版（没有文字层）")
else:
    total_chars = sum(len(d.text) for d in documents)
    print(f"[诊断] 总字符数：{total_chars}")
    print(f"[诊断] 第一段内容预览：\n{documents[0].text[:500]}\n")
    if total_chars < 200:
        print("[诊断] 警告：提取出的文字非常少，这份 PDF 很可能是扫描图片版，pypdf 无法提取文字，需要 OCR 方案")

# 5. 自动切片、向量化并建立索引（这就是 RAG 的核心步骤）
index = VectorStoreIndex.from_documents(documents)

# 6. 创建查询引擎，把 top_k 调大，让模型看到更多上下文（默认只取2段，信息量太少）
query_engine = index.as_query_engine(similarity_top_k=6)

print("RAG 系统构建成功！你可以开始提问了。")

# === 诊断：先看看检索到的内容是否相关、是否有实际信息 ===
your_question = "这个文档里提到的主要痛点/核心技术/团队成员有哪些？"

retriever = index.as_retriever(similarity_top_k=6)
retrieved_nodes = retriever.retrieve(your_question)
print(f"\n[诊断] 检索到 {len(retrieved_nodes)} 个相关片段：")
for i, node in enumerate(retrieved_nodes):
    print(f"--- 片段 {i+1} (相关度 {node.score:.3f}) ---")
    print(node.text[:300])
    print()

# === 正式提问，加强 prompt 要求具体内容，避免空话 ===
prompt = f"""Please answer the question based on the document content. Requirements:
1. You must cite specific names, numbers, and terms found in the document — avoid vague words like "some" or "certain"
2. If the document does not mention something, explicitly say "Not mentioned in the document" — do not make things up
3. Present the answer in a clear bullet-point format
4. Answer in English, regardless of the language of the source document

Question: {your_question}"""

response = query_engine.query(prompt)
print("\n[Final Answer]")
print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

基础模型配置完成！开始读取数据...

[诊断] 共读取到 91 个文档片段
[诊断] 总字符数：97016
[诊断] 第一段内容预览：
 
 
 
 
 
 
 
FAKULTI KECERDASAN BUATAN DAN KESELAMATAN SIBER 
 
SEMESTER 1 2025/2026 
 
WORKSHOP 2 (BAXU 3923) 
 
BAXI 
 
FINAL REPORT 
 
PROJECT TITLE: 
PERSONALIZED NUTRITIONAL MEAL PLANNING AND FOOD WASTE 
REDUCTION SYSTEM 
 
GROUP NUMBER: 
26 
 
PREPARED BY: 
HO RONG JIE (B032310541) 
MUHAMMAD AKMAL BAIHAQI BIN KHAIRUL ANAM MAK (B032310066) 
FATHIN NURQASDINA BINTI RASHID (B032410331) 
DHESEETHRA A/P BALAKRISHNAN (B032310621) 
 
This report is submitted in partial fulfilment of the requirem

RAG 系统构建成功！你可以开始提问了。

[诊断] 检索到 6 个相关片段：
--- 片段 1 (相关度 0.667) ---
iv 
 
 
ABSTRAK 
 
              Projek ini membentangkan pembangunan sistem pintar yang berfokuskan 
kepada cadangan hidangan yang peribadi dan pengurangan sisa makanan. Pada masa 
kini, sisa makanan isi rumah kekal menjadi cabaran global yang signifikan yang 
sering kali diburukkan lagi oleh peran

--- 片段 2 (相关度 0.666) ---
v 
 
TABLE OF CONTENTS 
 
 
ACKNOWLED